In [14]:
PDF_FILE = "DOCKER.pdf"

MODEL = "qwen3:0.6b"

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_FILE)
pages = loader.load()

In [16]:
print(f"No. of Pages: {len(pages)}")
print(f"Length of a page: {len(pages[1].page_content)}")

No. of Pages: 6
Length of a page: 1592


In [17]:
print("Content of a page:", pages[1].page_content)

Content of a page: Page 2 of 6 
Why are containers light weight ? Containers are lightweight because they use a technology called containerization, which allows them to share the host operating system's kernel and libraries, while still providing isolation for the application and its dependencies. This results in a smaller footprint compared to traditional virtual machines, as the containers do not need to include a full operating system. Additionally, Docker containers are designed to be minimal, only including what is necessary for the application to run, further reducing their size. Let's try to understand this with an example: Below is the screenshot of o@icial ubuntu base image which you can use for your container. It's just ~ 22 MB, isn't it very small ? on a contrary if you look at o@icial ubuntu VM image it will be close to ~ 2.3 GB. So the container base image is almost 100 times less than VM image. 
 To provide a better picture of ﬁles and folders that containers base images 

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)

chunks = splitter.split_documents(pages)

In [8]:
print(f"Number of chunks: {len(chunks)}")
print(f"Length of a chunk: {len(chunks[1].page_content)}")

Number of chunks: 11
Length of a chunk: 902


In [15]:
print("Content of a chunk:", chunks[1].page_content)

Content of a chunk: quickly. They bring great benefits, but also real and growing risks.
Suleyman is wise enough to know that there’s no simple three-point plan
for managing these risks, and brave enough to tell us so. This book is
honest, passionate, and unafraid to confront what is clearly one of the great
challenges our species will face this century. Thanks to Suleyman we know
what the situation is and what our options are. Now it’s up to us to act.”
—Andrew McAfee, principal research scientist at MIT Sloan,
author of The Geek Way
“The AI revolution is underway, but how well do we really understand it?
The Coming Wave offers an erudite, clear-eyed guide both to the history of
radical technological change and to the deep political challenges that lie
ahead.”
—Anne Applebaum, Pulitzer Prize–winning historian
“When this landed in my inbox, I cleared the diary and got reading. This is
an extraordinary and necessary book; the awe-inspiring thought is that in
twenty years it will seem al

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)

In [19]:
retriever = vectorstore.as_retriever()
retriever.invoke("What can you do?")

[Document(id='9699ee01-6e4d-47ac-a475-b6dfcfdbb6fe', metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'DOCKER.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content="Page 2 of 6 \nWhy are containers light weight ? Containers are lightweight because they use a technology called containerization, which allows them to share the host operating system's kernel and libraries, while still providing isolation for the application and its dependencies. This results in a smaller footprint compared to traditional virtual machines, as the containers do not need to include a full operating system. Additionally, Docker containers are designed to be minimal, only including what is necessary for the application to run, further reducing their size. Let's try to understand this with an example: Below is the screenshot of o@icial ubuntu base image which you can use for your container. It's just ~ 22 MB, isn't it very small ? on a contrary if you look at o@icial

In [20]:
from langchain_ollama import ChatOllama

model = ChatOllama(model=MODEL, temperature=0)
model.invoke("How can I use Docker?")

AIMessage(content="<think>\nOkay, the user is asking how to use Docker. Let me start by explaining what Docker is. I remember it's a containerization tool that allows developers to package and run applications in a lightweight environment. So first, I should mention that Docker is used for building, running, and managing containers.\n\nI should explain the basics: using Docker to build images, run containers, and manage them. Maybe give an example like building a web app with Docker. Then, the user might be a beginner, so I need to make sure the explanation is simple and not too technical. Also, mention that Docker can be used for different purposes like development, testing, and deployment.\n\nWait, the user might not know how to install Docker. So I should include steps on how to install it on their system. Maybe mention that they need to install Docker on their machine, then run the command. Also, explain that Docker can be used in various environments, like local development, cloud

In [21]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser
print(chain.invoke("How can I use Docker?"))

<think>
Okay, the user is asking how to use Docker. Let me start by explaining the basics. First, Docker is a platform that allows developers to package and run applications in a containerized environment. So I should mention that it's a containerization tool.

I need to explain what a Docker image is. Maybe start with an example, like a simple web app. Then talk about the Dockerfile, which defines the application's source code. Also, the Docker Hub is where images are stored. Oh right, and the Dockerfile is a text file that specifies the environment, dependencies, etc.

Then, the user might be a beginner, so I should break it down step by step. Maybe start with installing Docker on their system. Then, creating a new image, building it, and running it. Also, mention that Docker can be used for different types of applications, like web apps, databases, etc.

Wait, the user might not know the steps. So I should outline the process clearly. Maybe list the steps: install Docker, create a D

In [22]:
from langchain.prompts import PromptTemplate

template = """
You are an assistant that provides answers to questions based on
a given context. 

Answer the question based on the context. If you can't answer the
question, reply "I don't know".

Be as concise as possible and go straight to the point.

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is some context", question="Here is a question"))


You are an assistant that provides answers to questions based on
a given context. 

Answer the question based on the context. If you can't answer the
question, reply "I don't know".

Be as concise as possible and go straight to the point.

Context: Here is some context

Question: Here is a question



In [23]:
chain = prompt | model | parser

chain.invoke(
    {"context": "Anna's sister is Susan", "question": "Who is Susan's sister?"}
)

"<think>\nOkay, let's see. The context says Anna's sister is Susan. The question is asking who Susan's sister is. So if Anna's sister is Susan, then Susan's sister would be Anna. That makes sense because the relationship is a direct sibling pair. I don't think there's any other information needed here. Just need to state that Susan's sister is Anna.\n</think>\n\nAnna's sister is Susan. Therefore, Susan's sister is Anna."

In [24]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

In [25]:
questions = [
    "Docker is a tool for what?",
    "Run commands in a Docker container",
    "How to kill a Docker container",
]

for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: Docker is a tool for what?
Answer: <think>
Okay, the user is asking what Docker is for, based on the provided context. Let me look through the documents.

First, the context mentions Docker as a containerization platform that helps build container images, run containers, and push images to registries. Then, in the next document, they talk about Docker's architecture and lifecycle, like Docker Daemon being the brain. The last document explains lightweight because they share host OS and use minimal components. 

The question is straightforward: Docker is a tool for containerization. The answer should be concise. I need to check if any other parts mention Docker's purpose. The context does mention building images, running containers, and sharing images. So the answer is clear. No need to add extra info. Just state that Docker is a containerization tool.
</think>

Docker is a containerization platform that helps build and run container images.
*************************

Question: